# Day 13 — Contextual Recall & Relevancy

**Module 3 · RAG Evaluation**

Yesterday we evaluated **ranking** with Contextual Precision.

Today we complete the retriever diagnostic:

| Metric | Question |
|---|---|
| Contextual Precision | Is relevant information ranked highly? |
| Contextual Recall | Did we retrieve the information we needed? |
| Contextual Relevancy | How much irrelevant information did we retrieve? |

Together, these metrics help us understand **retrieval quality before the LLM generates an answer**.

In [1]:
import os

from dotenv import load_dotenv
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.models import OpenAIModel
from deepeval.metrics import (
    ContextualRecallMetric,
    ContextualRelevancyMetric,
)
from deepeval.test_case import LLMTestCase

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."

judge = OpenAIModel(
    model="gpt-4.1-mini",
    temperature=0,
)

async_config = AsyncConfig(max_concurrent=2)

print("Judge:", judge.get_model_name())

Judge: gpt-4.1-mini


## 1. Contextual Recall

Contextual Recall asks:

> **"Does the retrieved context contain the information needed to produce the expected answer?"**

Unlike Precision, **position does not matter**.

If the correct information is present anywhere in the retrieved context, recall can be high.

In [2]:
question = "How does RAG reduce hallucination?"

expected_answer = (
    "RAG reduces hallucination by retrieving relevant information "
    "and providing it to the LLM as context."
)

relevant_context = (
    "RAG grounds an LLM's answer in external documents. "
    "The system retrieves relevant information and provides it to the LLM "
    "as context, which can reduce hallucinations."
)

irrelevant_context = (
    "An embedding is a vector representation of text. "
    "Texts with similar meanings have vectors that are close together."
)

case_good = LLMTestCase(
    input=question,
    expected_output=expected_answer,
    retrieval_context=[relevant_context],
)

case_missing = LLMTestCase(
    input=question,
    expected_output=expected_answer,
    retrieval_context=[irrelevant_context],
)

## 2. Run Contextual Recall

The first case contains the information needed for the expected answer.

The second case does not.

This lets us isolate **retrieval coverage**.

In [3]:
recall = ContextualRecallMetric(
    model=judge,
    threshold=0.5,
)

results = evaluate(
    test_cases=[case_good, case_missing],
    metrics=[recall],
    async_config=async_config,
)

for result in results.test_results:
    metric_result = result.metrics_data[0]

    print(
        f"{result.name}: "
        f"score={metric_result.score:.2f}, "
        f"success={metric_result.success}"
    )
    print(f"reason: {metric_result.reason}\n")

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              How does RAG reduce hallucination?                                                   │
│  │     Actual Output:      None                                                                                 │
│  │     Expected Output:    RAG reduces hallucination by retrieving relevant information and providing it to     │
│  │                         the LLM as context.                                                                  │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric            ┃ Score ┃ Threshold ┃ Reason                                                   │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Contextual Recall │ 0.00  │ 0.50      │ The score is 0.00 because the expected output about      │
│              │                   │       │           │ RAG reducing hallucination is not supported by any       │
│              │                   │       │           │ information in the retrieval context, which only         │
│              │                   │       │           │ discusses embeddings as vector representations of text   │
│              │                   │       │           │ without mentioning RAG or hallucination reduction.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                      ┃ Average Score         ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Contextual Recall           │ 0.50                  │ 50.00% | passed=1 | failed=1                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=885881;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 6.28s | token cost: 0.0010096000000000003 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_case_1: score=0.00, success=False
reason: The score is 0.00 because the expected output about RAG reducing hallucination is not supported by any information in the retrieval context, which only discusses embeddings as vector representations of text without mentioning RAG or hallucination reduction.

test_case_0: score=1.00, success=True
reason: The score is 1.00 because the first sentence in the expected output is fully supported by node 1 in the retrieval context, which explicitly states that the system retrieves relevant information and provides it to the LLM as context, reducing hallucinations. There are no unsupportive reasons, indicating complete alignment.



## 3. Contextual Relevancy

Now consider a different problem.

Suppose the correct document was retrieved, but the retriever also returned several unrelated documents.

Contextual Relevancy asks:

> **"How relevant is the retrieved context to the question?"**

This measures the **signal-to-noise ratio** of retrieval.

In [4]:
case_clean = LLMTestCase(
    input=question,
    expected_output=expected_answer,
    retrieval_context=[
        relevant_context,
    ],
)

case_noisy = LLMTestCase(
    input=question,
    expected_output=expected_answer,
    retrieval_context=[
        "Llamas are domesticated South American camelids.",
        "The capital of France is Paris.",
        "Python is a popular programming language.",
        relevant_context,
    ],
)

In [5]:
relevancy = ContextualRelevancyMetric(
    model=judge,
    threshold=0.5,
)

results = evaluate(
    test_cases=[case_clean, case_noisy],
    metrics=[relevancy],
    async_config=async_config,
)

for result in results.test_results:
    metric_result = result.metrics_data[0]

    print(
        f"{result.name}: "
        f"score={metric_result.score:.2f}, "
        f"success={metric_result.success}"
    )
    print(f"reason: {metric_result.reason}\n")

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              How does RAG reduce hallucination?                                                   │
│  │     Actual Output:      None                                                                                 │
│  │     Expected Output:    RAG reduces hallucination by retrieving relevant information and providing it to     │
│  │                         the LLM as context.                                                                  │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Contextual Relevancy │ 0.40  │ 0.50      │ The score is 0.40 because while most statements       │
│              │                      │       │           │ like 'Llamas are domesticated South American          │
│              │                      │       │           │ camelids.' are irrelevant to how RAG reduces          │
│              │                      │       │           │ hallucination, the relevant statements 'RAG grounds   │
│              │                      │       │           │ an LLM's answer in external documents.' and 'The      │
│              │                      │       │           │ system retrieves relevant information and provides    │
│              │                      │       │           │ it to the LLM as context, which can reduce            │
│              │                      │       │           │ hallucinations.' directly address the input           │
│              │                      │       │           │ question.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                          ┃ Average Score        ┃ Pass Rate                                 ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Contextual Relevancy            │ 0.70                 │ 50.00% | passed=1 | failed=1              │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=648782;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.44s | token cost: 0.00198 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_case_0: score=1.00, success=True
reason: The score is 1.00 because the relevant statements clearly explain that RAG reduces hallucination by grounding an LLM's answer in external documents and providing relevant information as context, directly addressing how RAG achieves this.

test_case_1: score=0.40, success=False
reason: The score is 0.40 because while most statements like 'Llamas are domesticated South American camelids.' are irrelevant to how RAG reduces hallucination, the relevant statements 'RAG grounds an LLM's answer in external documents.' and 'The system retrieves relevant information and provides it to the LLM as context, which can reduce hallucinations.' directly address the input question.



## 4. The Retriever Diagnostic

We now have three complementary questions:

| Metric | What it tells us |
|---|---|
| **Precision** | Is useful information near the top? |
| **Recall** | Did we retrieve the information at all? |
| **Relevancy** | Did we retrieve unnecessary information? |

Think of them as:

```text
Precision → POSITION
Recall    → COVERAGE
Relevancy → NOISE

# Day 13 — Key Takeaways

- **Contextual Recall** measures whether the retrieved context contains the information needed for the expected answer.
- **Contextual Relevancy** measures how much of the retrieved context is useful for the question.
- Recall is about **coverage**.
- Relevancy is about **noise**.
- Together with Contextual Precision, they give us a useful picture of retriever quality.

### Retriever evaluation

```text
                 RETRIEVER
                    │
       ┌────────────┼────────────┐
       ↓            ↓            ↓
   Precision      Recall      Relevancy
    Position     Coverage       Noise